In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pyspark h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00


In [ ]:
from pyspark.sql import SparkSession

# Initialize the Spark Session
spark = SparkSession.builder \
    .appName("Uber_Weather_Hotspot_Analysis") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Spark is successfully running!")

Spark is successfully running!


In [ ]:
# Define the file paths (Update these if your folder or file names are different!)
uber_path = "/content/drive/MyDrive/Big Data Project/uber-raw-data-apr14.csv"
weather_path = "/content/drive/MyDrive/Big Data Project/weather_description.csv"

# Load the data into Spark DataFrames
uber_df = spark.read.csv(uber_path, header=True, inferSchema=True)
weather_df = spark.read.csv(weather_path, header=True, inferSchema=True)

# Print the schema (the columns) and the first 20 rows to prove it worked
print("--- Uber Data ---")
uber_df.printSchema()
uber_df.show()

print("--- Weather Data ---")
weather_df.show()

--- Uber Data ---
root
 |-- Date/Time: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Lon: double (nullable = true)
 |-- Base: string (nullable = true)

+----------------+-------+--------+------+
|       Date/Time|    Lat|     Lon|  Base|
+----------------+-------+--------+------+
|4/1/2014 0:11:00| 40.769|-73.9549|B02512|
|4/1/2014 0:17:00|40.7267|-74.0345|B02512|
|4/1/2014 0:21:00|40.7316|-73.9873|B02512|
|4/1/2014 0:28:00|40.7588|-73.9776|B02512|
|4/1/2014 0:33:00|40.7594|-73.9722|B02512|
|4/1/2014 0:33:00|40.7383|-74.0403|B02512|
|4/1/2014 0:39:00|40.7223|-73.9887|B02512|
|4/1/2014 0:45:00| 40.762| -73.979|B02512|
|4/1/2014 0:55:00|40.7524| -73.996|B02512|
|4/1/2014 1:01:00|40.7575|-73.9846|B02512|
|4/1/2014 1:19:00|40.7256|-73.9869|B02512|
|4/1/2014 1:48:00|40.7591|-73.9684|B02512|
|4/1/2014 1:49:00|40.7271|-73.9803|B02512|
|4/1/2014 2:11:00|40.6463|-73.7896|B02512|
|4/1/2014 2:25:00|40.7564|-73.9167|B02512|
|4/1/2014 2:31:00|40.7666|-73.9531|B02512|
|4/1/2014 2:

In [ ]:
# The '*' tells Spark to load EVERY file that ends in '14.csv'
# This automatically combines April, May, June, July, August, and September into one massive dataset.
uber_path = "/content/drive/MyDrive/Big Data Project/uber-raw-data-*14.csv"

# Load the data
uber_df = spark.read.csv(uber_path, header=True, inferSchema=True)

# Count the exact number of rows to prove it loaded millions
row_count = uber_df.count()
print(f"Total Uber trips loaded: {row_count:,}")

Total Uber trips loaded: 4,534,327


In [ ]:
import h3
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# 1. Define the function to convert Latitude and Longitude into an H3 Hexagon ID
# Resolution 9 creates hexagons roughly the size of a few city blocks.
def get_h3_hex(lat, lon):
    try:
        # We convert the coordinates to a float just to be safe
        return h3.latlng_to_cell(float(lat), float(lon), 9)
    except:
        return None

# 2. Register this function so Apache Spark can use it across millions of rows
hex_udf = udf(get_h3_hex, StringType())

# 3. Apply the function to create a brand new column called 'hex_id'
print("Calculating geospatial hexagons...")
uber_hex_df = uber_df.withColumn("hex_id", hex_udf(col("Lat"), col("Lon")))

# 4. Show the results to prove it worked!
uber_hex_df.select("Date/Time", "Lat", "Lon", "hex_id").show()

Calculating geospatial hexagons...
+----------------+-------+--------+---------------+
|       Date/Time|    Lat|     Lon|         hex_id|
+----------------+-------+--------+---------------+
|9/1/2014 0:01:00|40.2201|-74.0021|892a1382177ffff|
|9/1/2014 0:01:00|  40.75|-74.0027|892a107252fffff|
|9/1/2014 0:03:00|40.7559|-73.9864|892a100d66bffff|
|9/1/2014 0:06:00| 40.745|-73.9889|892a100d21bffff|
|9/1/2014 0:11:00|40.8145|-73.9444|892a1008cb7ffff|
|9/1/2014 0:12:00|40.6735|-73.9918|892a1077453ffff|
|9/1/2014 0:15:00|40.7471|-73.6472|892a10044a3ffff|
|9/1/2014 0:16:00|40.6613|-74.2691|892a1078d4bffff|
|9/1/2014 0:32:00|40.3745|-73.9999|892a1393313ffff|
|9/1/2014 0:33:00|40.7633|-73.9773|892a100d61bffff|
|9/1/2014 0:33:00|40.7467|-73.6131|892a100616bffff|
|9/1/2014 0:37:00|40.8105|  -73.96|892a1008c3bffff|
|9/1/2014 0:38:00| 40.679|-74.0111|892a107296fffff|
|9/1/2014 0:39:00|40.4023|-73.9839|892a139346fffff|
|9/1/2014 0:48:00|40.7378|-74.0395|892a10723abffff|
|9/1/2014 0:48:00|40.7214|-73

In [ ]:
from pyspark.sql.functions import to_timestamp, date_trunc, col

print("Cleaning and formatting timestamps...")

# 1. Format the Uber data
# We convert the text date into a real Timestamp, then truncate it to the nearest hour
uber_prepped = uber_hex_df.withColumn("timestamp", to_timestamp(col("Date/Time"), "M/d/yyyy H:mm:ss")) \
                          .withColumn("join_hour", date_trunc("hour", col("timestamp")))

# 2. Format the Weather data
# We only want the datetime and New York columns. We rename "New York" to "weather_condition"
weather_prepped = weather_df.select(
    to_timestamp(col("datetime"), "yyyy-MM-dd HH:mm:ss").alias("join_hour"),
    col("New York").alias("weather_condition")
)

# Show the prepped weather data to ensure it looks right
weather_prepped.show()

Cleaning and formatting timestamps...
+-------------------+-----------------+
|          join_hour|weather_condition|
+-------------------+-----------------+
|2012-10-01 12:00:00|             NULL|
|2012-10-01 13:00:00|       few clouds|
|2012-10-01 14:00:00|       few clouds|
|2012-10-01 15:00:00|       few clouds|
|2012-10-01 16:00:00|       few clouds|
|2012-10-01 17:00:00|       few clouds|
|2012-10-01 18:00:00|       few clouds|
|2012-10-01 19:00:00|       few clouds|
|2012-10-01 20:00:00|       few clouds|
|2012-10-01 21:00:00|       few clouds|
|2012-10-01 22:00:00|       few clouds|
|2012-10-01 23:00:00|       few clouds|
|2012-10-02 00:00:00|       few clouds|
|2012-10-02 01:00:00|       few clouds|
|2012-10-02 02:00:00|       few clouds|
|2012-10-02 03:00:00|       few clouds|
|2012-10-02 04:00:00|       few clouds|
|2012-10-02 05:00:00|       few clouds|
|2012-10-02 06:00:00|       few clouds|
|2012-10-02 07:00:00|       few clouds|
+-------------------+-----------------+
on

In [ ]:
print("Joining Uber and Weather datasets...")

# Perform an Inner Join based on the 'join_hour' column
final_joined_df = uber_prepped.join(weather_prepped, on="join_hour", how="inner")

# Drop the trips where the weather data was missing (null) to keep our data clean
final_joined_df = final_joined_df.na.drop(subset=["weather_condition"])

# Show the final result!
final_joined_df.select("join_hour", "Lat", "Lon", "hex_id", "weather_condition").show()

Joining Uber and Weather datasets...
+-------------------+-------+--------+---------------+-----------------+
|          join_hour|    Lat|     Lon|         hex_id|weather_condition|
+-------------------+-------+--------+---------------+-----------------+
|2014-09-01 00:00:00|40.2201|-74.0021|892a1382177ffff|             mist|
|2014-09-01 00:00:00|  40.75|-74.0027|892a107252fffff|             mist|
|2014-09-01 00:00:00|40.7559|-73.9864|892a100d66bffff|             mist|
|2014-09-01 00:00:00| 40.745|-73.9889|892a100d21bffff|             mist|
|2014-09-01 00:00:00|40.8145|-73.9444|892a1008cb7ffff|             mist|
|2014-09-01 00:00:00|40.6735|-73.9918|892a1077453ffff|             mist|
|2014-09-01 00:00:00|40.7471|-73.6472|892a10044a3ffff|             mist|
|2014-09-01 00:00:00|40.6613|-74.2691|892a1078d4bffff|             mist|
|2014-09-01 00:00:00|40.3745|-73.9999|892a1393313ffff|             mist|
|2014-09-01 00:00:00|40.7633|-73.9773|892a100d61bffff|             mist|
|2014-09-01 00

In [ ]:
import pyspark.sql.functions as F

print("Aggregating trips by Hexagon and Weather...")

# Group by the hexagon ID and the weather condition, then count the trips
hotspot_df = final_joined_df.groupBy("hex_id", "weather_condition") \
                            .agg(F.count("*").alias("trip_count"))

# Sort it from highest to lowest to see the busiest hotspots immediately
hotspot_df = hotspot_df.orderBy(F.desc("trip_count"))

# Show the top busiest combinations!
hotspot_df.show()

Aggregating trips by Hexagon and Weather...
+---------------+-----------------+----------+
|         hex_id|weather_condition|trip_count|
+---------------+-----------------+----------+
|892a103b1cbffff|    broken clouds|      9020|
|892a10721b3ffff|    broken clouds|      8822|
|892a100f52bffff|    broken clouds|      8626|
|892a100d603ffff|    broken clouds|      7608|
|892a100d61bffff|    broken clouds|      7560|
|892a10721b3ffff| scattered clouds|      7489|
|892a100d613ffff|    broken clouds|      7099|
|892a100d63bffff|    broken clouds|      6999|
|892a103b1cbffff| scattered clouds|      6971|
|892a1072c17ffff|    broken clouds|      6786|
|892a100d677ffff|    broken clouds|      6664|
|892a100f52bffff| scattered clouds|      6480|
|892a100d60fffff|    broken clouds|      6479|
|892a100d66bffff|    broken clouds|      6402|
|892a100d21bffff|    broken clouds|      6042|
|892a100d607ffff|    broken clouds|      6032|
|892a100d61bffff| scattered clouds|      6028|
|892a100d603ffff

In [ ]:
# Define where to save the final file in your Drive
output_path = "/content/drive/MyDrive/Big Data Project/uber_weather_hotspots.csv"

print("Saving final dataset to Google Drive...")

# We convert the Spark DataFrame to a Pandas DataFrame to easily save it as one clean CSV file
hotspot_df.toPandas().to_csv(output_path, index=False)

print(f"Saved successfully to: {output_path}")

Saving final dataset to Google Drive...
Saved successfully to: /content/drive/MyDrive/Big Data Project/uber_weather_hotspots.csv
